# TechMatch — free Colab demo

Run all cells (**Runtime → Run all**) to launch the app and get a **public URL**.
No account beyond Google, no credit card. The URL is **temporary** (per session).

CPU runtime is fine (Colab's bundled torch is reused, so install is fast).


In [ ]:
# Clone the (public) repo and install dependencies.
import os

REPO = "Ai-Job-Search"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/Xvbgf467/Ai-Job-Search.git
%cd -q {REPO}
!pip install -q . 2>&1 | tail -n 3
print("installed ->", os.getcwd())


In [ ]:
# Configure the Z.AI LLM re-ranker. The key is read at runtime only — never saved.
import os

os.environ["LLM_PROVIDER"]     = "zai"
os.environ["LLM_MODEL"]        = "glm-4.5-flash"
os.environ["LLM_BASE_URL"]     = "https://api.z.ai/api/paas/v4/"
os.environ["LLM_RERANK_TOP_N"] = "20"

key = ""
try:
    from google.colab import userdata
    key = userdata.get("LLM_API_KEY") or ""
    print("loaded key from Colab Secrets")
except Exception:
    from getpass import getpass
    key = getpass("Paste your Z.AI API key (blank to skip re-rank): ")

if key:
    os.environ["LLM_API_KEY"] = key
    print("LLM re-rank: ENABLED")
else:
    os.environ["LLM_PROVIDER"] = ""
    print("LLM re-rank: DISABLED (no key) — keyword+embedding matching still works")


In [ ]:
# Launch the API in the background.
import sys, subprocess, time, os

open("app.log", "w").close()
api_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "7860"],
    cwd=os.getcwd(),
    stdout=open("app.log", "a"),
    stderr=subprocess.STDOUT,
)
print("uvicorn pid:", api_proc.pid)
time.sleep(6)
print(open("app.log", errors="ignore").read()[-1500:])


In [ ]:
# Expose port 7860 via a free Cloudflare quick tunnel (no account, no token).
import subprocess, time, re

subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
subprocess.run(["wget", "-q",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "/usr/local/bin/cloudflared"], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

cf_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
                           stdout=open("cf.log", "w"), stderr=subprocess.STDOUT)

public_url = None
for _ in range(30):
    try:
        txt = open("cf.log", errors="ignore").read()
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
        if m:
            public_url = m.group(0); break
    except Exception:
        pass
    time.sleep(2)

print("PUBLIC URL:", public_url or "(not ready — re-run this cell)")


In [ ]:
# Wait for startup (model load + job seeding), then health-check the public URL.
import time, urllib.request

if public_url:
    for _ in range(40):
        try:
            with urllib.request.urlopen(public_url + "/health", timeout=5) as r:
                print("HEALTH:", r.status, r.read().decode()[:200]); break
        except Exception:
            time.sleep(3)
    else:
        print("still starting — tail of app.log:")
        print(open("app.log", errors="ignore").read()[-1000:])
    print("\nOpen the app UI:\n  ", public_url)
    print("API docs (Swagger):\n  ", public_url + "/docs")
else:
    print("no public URL — re-run the tunnel cell")


## Notes
- **Free & instant, but temporary**: the public URL and the app die when this Colab session disconnects/idles. Re-run the cells to restart (a new URL is minted each time).
- To keep your Z.AI key out of the notebook, add it once via the **\U0001F511 Secrets** icon on the left → name `LLM_API_KEY`. Otherwise you'll paste it in the prompt each run.
- To stop the app: `api_proc.terminate()`. To stop the tunnel: `cf_proc.terminate()`.
- Source: https://github.com/Xvbgf467/Ai-Job-Search
